In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

In [2]:
torch.manual_seed(42)

X = torch.randn(600, 2)

y = torch.zeros(600, dtype=torch.long)

y[(X[:, 0] > 0) & (X[:, 1] > 0)] = 1
y[X[:, 0] < 0] = 2

In [3]:
X_train = X[:500]
y_train = y[:500]

X_test = X[500:]
y_test = y[500:]

In [4]:
train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [5]:
class MulticlassANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),

            nn.Linear(16, 16),
            nn.ReLU(),

            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)

In [6]:
model = MulticlassANN()

loss_fn = nn.CrossEntropyLoss()

In [7]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [8]:
epochs = 50

for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        # Forward pass
        logits = model(X_batch)

        # Loss
        loss = loss_fn(logits, y_batch)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}, Loss: {loss.item():.4f}"
        )

Epoch 10, Loss: 0.3104
Epoch 20, Loss: 0.2337
Epoch 30, Loss: 0.0517
Epoch 40, Loss: 0.0345
Epoch 50, Loss: 0.0245


In [9]:
model.eval()

with torch.no_grad():

    logits = model(X_test)

    predictions = torch.argmax(logits, dim=1)

    accuracy = (
        (predictions == y_test)
        .float()
        .mean()
    )

print("Accuracy:", accuracy.item())

Accuracy: 0.9900000095367432


In [10]:
probabilities = torch.softmax(logits, dim=1)

In [11]:
probabilities

tensor([[1.1668e-05, 3.6890e-05, 9.9995e-01],
        [9.9659e-01, 3.4052e-03, 3.0502e-07],
        [8.5892e-05, 9.9991e-01, 2.9235e-06],
        [2.4285e-02, 9.7559e-01, 1.2907e-04],
        [9.9682e-01, 1.7381e-03, 1.4427e-03],
        [2.1520e-01, 7.6722e-01, 1.7580e-02],
        [9.9633e-01, 2.9382e-03, 7.2926e-04],
        [3.2054e-04, 9.9967e-01, 5.4335e-06],
        [9.9955e-01, 1.7791e-04, 2.7098e-04],
        [1.1065e-05, 1.8998e-06, 9.9999e-01],
        [2.6535e-03, 9.7295e-01, 2.4396e-02],
        [1.9087e-04, 9.9976e-01, 4.8039e-05],
        [6.4744e-07, 9.4478e-06, 9.9999e-01],
        [3.9579e-07, 1.7413e-06, 1.0000e+00],
        [5.3084e-02, 1.0401e-01, 8.4291e-01],
        [4.3308e-06, 2.5047e-07, 1.0000e+00],
        [2.3533e-03, 9.9764e-01, 1.0767e-05],
        [3.0315e-01, 6.0438e-02, 6.3642e-01],
        [1.3522e-06, 7.2362e-06, 9.9999e-01],
        [2.0816e-02, 1.6660e-01, 8.1258e-01],
        [1.3815e-03, 5.5603e-04, 9.9806e-01],
        [9.9999e-01, 4.6382e-06, 5

# 📘 Module 15.4 — Multiclass Classification ANN — Notes

## 1. Multiclass Classification 🔥🔥🔥

**Multiclass classification** means the model chooses **exactly one class from more than two possible classes**.

Examples:

* Digit classification → `0–9` → 10 classes
* Animal classification → Cat / Dog / Horse
* Object classification → Car / Bike / Bus

If there are `K` classes:

> **Output layer needs K neurons.**

---

# 2. ANN Architecture

For a 3-class problem:

```text
Input
  ↓
Linear(2 → 16)
  ↓
ReLU
  ↓
Linear(16 → 16)
  ↓
ReLU
  ↓
Linear(16 → 3)
  ↓
3 RAW LOGITS
```

Example output:

```text
[2.1, -0.5, 4.3]
```

These are called **logits**.

They are **not probabilities**.

---

# 3. Why K Output Neurons? 🔥

For `K` classes:

```text
Number of output neurons = K
```

Example:

| Classes |              Output neurons |
| ------: | --------------------------: |
|       2 | 1 for standard binary setup |
|       3 |                           3 |
|       5 |                           5 |
|      10 |                          10 |
|     100 |                         100 |

For MNIST:

```text
0 1 2 3 4 5 6 7 8 9
```

Therefore:

```python
nn.Linear(hidden_size, 10)
```

---

# 4. Logits

The final layer produces:

$$
z = Wx+b
$$

For 3 classes:

$$
z=[z_0,z_1,z_2]
$$

Example:

```text
[2.1, -0.5, 4.3]
```

The largest logit corresponds to the predicted class.

```text
2.1  → Class 0
-0.5 → Class 1
4.3  → Class 2  ← prediction
```

Therefore:

```python
prediction = torch.argmax(logits, dim=1)
```

---

# 5. Softmax 🔥🔥🔥

Softmax converts logits into probabilities:

$$
P_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Example:

```text
Logits
[2.1, -0.5, 4.3]

       ↓ Softmax

Probabilities
[0.10, 0.01, 0.89]
```

Properties:

* Each probability is between 0 and 1.
* Probabilities sum to 1.
* Largest logit → largest probability.

$$
\sum_iP_i=1
$$

### PyTorch

```python
probabilities = torch.softmax(logits, dim=1)
```

---

# 6. Critical PyTorch Rule 🚨

When using:

```python
nn.CrossEntropyLoss()
```

the model should normally output **raw logits**.

### Correct

```text
Model
 ↓
Raw logits
 ↓
CrossEntropyLoss
```

### Don't normally do

```text
Model
 ↓
Softmax
 ↓
CrossEntropyLoss
```

Because `CrossEntropyLoss` internally performs the required log-softmax operation together with the classification loss.

So:

```python
model = nn.Sequential(
    nn.Linear(16, 3)
)

loss_fn = nn.CrossEntropyLoss()
```

---

# 7. Target Labels 🔥🔥🔥

For multiclass classification using `CrossEntropyLoss`, targets are normally **integer class indices**.

Example:

```python
y = torch.tensor([0, 2, 1, 0, 2])
```

Meaning:

```text
0 → Class 0
2 → Class 2
1 → Class 1
0 → Class 0
2 → Class 2
```

Target dtype:

```python
torch.long
```

Usually:

```python
y = y.long()
```

### Important

You normally do **not** need one-hot targets:

```text
[1,0,0]
[0,0,1]
[0,1,0]
```

for the standard `CrossEntropyLoss` class-index setup.

---

# 8. Multiclass ANN Implementation

### Imports

```python
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
```

### Model

```python
class MulticlassANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),

            nn.Linear(16, 16),
            nn.ReLU(),

            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)
```

Architecture:

```text
Input: 2
   ↓
16 neurons
   ↓ ReLU
16 neurons
   ↓ ReLU
3 neurons
   ↓
3 logits
```

---

# 9. Loss Function

```python
loss_fn = nn.CrossEntropyLoss()
```

Conceptually:

```text
3 logits
   ↓
CrossEntropyLoss
   ↓
Loss
```

The loss encourages the correct class to receive a larger logit relative to the other classes.

---

# 10. Optimizer

Example:

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

Training components:

```text
Model
+
Loss
+
Optimizer
+
Data
=
Training
```

---

# 11. Training Loop 🔥🔥🔥

```python
for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        # Forward
        logits = model(X_batch)

        # Loss
        loss = loss_fn(logits, y_batch)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()
```

### Complete flow

```text
X
 ↓
Forward Pass
 ↓
Logits
 ↓
CrossEntropyLoss
 ↓
Loss
 ↓
Backward()
 ↓
Gradients
 ↓
Optimizer.step()
 ↓
Updated Parameters
```

---

# 12. Tensor Shapes 🔥🔥🔥

Suppose:

```text
Batch size = 32
Input features = 2
Classes = 3
```

Input:

```text
X → (32, 2)
```

After first layer:

```text
(32, 2)
   ↓
Linear(2,16)
   ↓
(32,16)
```

After second layer:

```text
(32,16)
   ↓
Linear(16,16)
   ↓
(32,16)
```

Output:

```text
(32,16)
   ↓
Linear(16,3)
   ↓
(32,3)
```

Target:

```text
y → (32,)
```

### Remember

```text
Input  → (B, features)
Output → (B, classes)
Target → (B,)
```

where `B` = batch size.

---

# 13. Prediction

After training:

```python
model.eval()

with torch.no_grad():

    logits = model(X_test)

    predictions = torch.argmax(
        logits,
        dim=1
    )
```

Example:

```text
logits:

[2.1, -0.5, 4.3]
[5.2,  1.1, -2.0]
[-1.2, 4.5, 0.3]
```

`argmax`:

```text
[2, 0, 1]
```

Therefore:

```text
Sample 1 → Class 2
Sample 2 → Class 0
Sample 3 → Class 1
```

---

# 14. Why Argmax Without Softmax?

Important concept:

```text
logits:
[2.1, -0.5, 4.3]

largest = 4.3
```

Softmax preserves the ordering of values.

Therefore:

```python
torch.argmax(logits, dim=1)
```

gives the same class as:

```python
torch.argmax(
    torch.softmax(logits, dim=1),
    dim=1
)
```

So **Softmax is not required just to get the predicted class**.

Use it when you actually need probabilities.

---

# 15. Accuracy

```python
accuracy = (
    (predictions == y_test)
    .float()
    .mean()
)
```

Example:

```text
100 test samples
85 correct

Accuracy = 85%
```

Formula:

$$
Accuracy=
\frac{\text{Correct Predictions}}
{\text{Total Predictions}}
$$

---

# 16. Binary vs Multiclass 🔥🔥🔥

| Feature              | Binary            | Multiclass       |
| -------------------- | ----------------- | ---------------- |
| Number of classes    | 2                 | K > 2            |
| Output               | 1 logit           | K logits         |
| Probability function | Sigmoid           | Softmax          |
| Standard loss        | BCEWithLogitsLoss | CrossEntropyLoss |
| Target               | 0/1               | 0 to K−1         |
| Prediction           | Threshold         | Argmax           |

### Memory trick 🧠

> **Binary:** 1 neuron → sigmoid conceptually → BCE
> **Multiclass:** K neurons → softmax conceptually → CE

---

# 17. Common Mistakes 🚨

### ❌ 1. Softmax + CrossEntropyLoss

```python
nn.Softmax(dim=1)
nn.CrossEntropyLoss()
```

Normally unnecessary.

Use raw logits with `CrossEntropyLoss`.

---

### ❌ 2. Using BCEWithLogitsLoss

For ordinary single-label multiclass classification:

```python
nn.BCEWithLogitsLoss()
```

is not the standard choice.

Use:

```python
nn.CrossEntropyLoss()
```

---

### ❌ 3. Wrong number of output neurons

3 classes:

```python
nn.Linear(16, 3)
```

Not:

```python
nn.Linear(16, 1)
```

---

### ❌ 4. Wrong target type

Use class indices:

```python
tensor([0, 2, 1, 0])
```

with integer/`long` dtype.

---

### ❌ 5. Forgetting `argmax`

```python
predictions = torch.argmax(logits, dim=1)
```

---

### ❌ 6. Confusing `eval()` and `no_grad()`

```python
model.eval()
```

changes model behavior for layers such as Dropout and BatchNorm.

```python
torch.no_grad()
```

disables gradient tracking.

They solve different problems.

---

# 18. Training vs Inference

### Training

```text
Input
 ↓
ANN
 ↓
Raw logits
 ↓
CrossEntropyLoss
 ↓
Backward
 ↓
Optimizer
 ↓
Update
```

### Inference

```text
Input
 ↓
ANN
 ↓
Raw logits
 ↓
Argmax
 ↓
Predicted class
```

If probabilities are required:

```text
Raw logits
 ↓
Softmax
 ↓
Probabilities
```

---

# 19. Industry / Practical Relevance 🔥

Multiclass ANN is the basic pattern behind many classification systems:

```text
Input features
      ↓
Neural Network
      ↓
K logits
      ↓
Class prediction
```

Examples:

* Image classification
* Digit recognition
* Document classification
* Intent classification
* Product category classification
* Fault-category classification
* Basic medical/image classification systems

For large image problems, however, the ANN is usually replaced by architectures such as CNNs or modern vision models.

---

# 20. Interview Points 🎯

### Q1. Why are there K output neurons?

Because each output neuron represents one class's score/logit.

### Q2. Why not apply Softmax before `CrossEntropyLoss`?

Because `CrossEntropyLoss` internally combines log-softmax with the negative log-likelihood computation.

### Q3. What does the model output?

**Raw logits**, not probabilities.

### Q4. How do you get the predicted class?

```python
torch.argmax(logits, dim=1)
```

### Q5. How do you get probabilities?

```python
torch.softmax(logits, dim=1)
```

### Q6. What should the target contain?

Integer class indices:

```text
0, 1, 2, ..., K-1
```

### Q7. Binary vs multiclass?

```text
Binary:
1 output → BCEWithLogitsLoss

Multiclass:
K outputs → CrossEntropyLoss
```

---

# 🧠 Final Cheat Sheet

```text
MULTICLASS ANN
────────────────────────────────

Problem:
Choose exactly ONE class from K classes.

Architecture:
Input → Hidden Layers → K outputs

Output:
K RAW LOGITS

Loss:
CrossEntropyLoss

Target:
Integer class indices
0 ... K-1

Training:
logits
  ↓
CrossEntropyLoss
  ↓
backward()
  ↓
optimizer.step()

Prediction:
argmax(logits)

Probabilities:
softmax(logits)

Shapes:
Input  → (B, features)
Output → (B, K)
Target → (B,)

DO NOT normally:
Softmax → CrossEntropyLoss
```

### 🔥 One line to remember

> **Multiclass ANN = K output logits + integer class labels + CrossEntropyLoss + argmax for prediction.**

